# 03 — Expanding Dimensionality

So far the solution has been a curve: one intensity as a function of one coordinate. Real radiative transfer problems are not curves. They are fields over several variables, and the reason to solve them is almost always to *look at* the result — a map, an image, a spectrum.

This notebook takes the problem you built in Notebook 00 and grows it along two independent axes, one per path. The paths do not depend on each other; you can do them in either order.

- **Path A — a second spatial dimension.** Move from a slab to a spherical shell, where the intensity depends on radius $\hat{r}$ *and* on the direction $\theta$ a ray is travelling. The solution becomes a two-dimensional field.
- **Path B — a wavelength axis.** Let the opacity and the source function depend on wavelength, so that one training run predicts a whole spectrum instead of a single measurement.

Both are the same move, made twice: **give the network another input, add the corresponding term to the residual, and plot the result.** Everything else — the hard boundary constraint, the training loop, the collocation idea — carries over unchanged from Notebooks 00 and 02.

## Learning goals

- add an input dimension to a PINN and to its residual;
- recognise which new terms a coordinate change introduces into a transport equation;
- identify the region of a two-dimensional domain that a boundary condition actually determines;
- build a wavelength-dependent source function and opacity from physical quantities you already have;
- produce and read the two standard visualisations of an expanded solution: a field map and an emergent spectrum.

Expected working time: about 2–2.5 hours.

## 1. Inherit your Notebook 00 solutions

As in Notebook 02, nothing gets pasted. The cell below runs your completed `00_setting_up_the_problem.ipynb` and picks up what you built there. Two pieces do most of the work in this notebook:

- **`coordinate_derivative`** — you wrote it for a single coordinate, but it needs no modification at all. Hand it a coordinate tensor with two columns and `torch.autograd.grad` returns two columns: the partial derivative with respect to each input. That is the whole mechanical content of "going to 2D".
- **`blackbody_lambda_cgs`** — Path B needs the Planck function at many wavelengths rather than at one, which is exactly what this already computes.

Also inherited: `SOURCE`, `u_0`, and `OPACITY` (your baseline optical depth $\tau$).

If the load fails, the traceback comes from Notebook 00: finish and save that notebook first. Instructors can point the filename at `00_setting_up_the_problem_instructor.ipynb`.

In [ ]:
from IPython.utils.capture import capture_output

with capture_output():
    get_ipython().run_line_magic("run", "00_setting_up_the_problem.ipynb")

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

torch.set_default_dtype(torch.float32)
torch.set_num_threads(1)

print("inherited from Notebook 00:")
print(f"  SOURCE  = {SOURCE}")
print(f"  u_0     = {u_0}")
print(f"  OPACITY = {OPACITY}   (baseline optical depth tau)")

# coordinate_derivative already handles more than one input column:
_probe = torch.tensor([[0.7, 0.3]], requires_grad=True)
_value = (_probe[:, 0:1] ** 2) * _probe[:, 1:2]          # f = a^2 * b
_grad = coordinate_derivative(_value, _probe)
print(f"\n  d/d(a,b) of a^2*b at (0.7, 0.3) -> {_grad.detach().tolist()[0]}")
print(f"  expected [2ab, a^2]            -> {[2*0.7*0.3, 0.7**2]}")

### Supplied components

The machinery below is the same as in Notebook 02, with one change: `MLP` now takes an `input_dim`, and the hard-boundary wrapper applies its constraint to the **first** input column. Read it once, then move on — none of it is an exercise.

In [ ]:
import random


def set_seed(seed: int) -> None:
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))


def make_generator(seed: int) -> torch.Generator:
    return torch.Generator(device="cpu").manual_seed(int(seed))


class MLP(nn.Module):
    """Tanh network mapping `input_dim` coordinates to one intensity."""

    def __init__(self, input_dim: int = 2, hidden_dim: int = 32, hidden_layers: int = 3) -> None:
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, 1)]
        self.network = nn.Sequential(*layers)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return self.network(X)


class HardBoundaryModel(nn.Module):
    """u = u_in + g(first column) * N(X), so u = u_in on the inflow boundary.

    The same trick as Notebook 02, now for a multi-column input: `g` must vanish
    at the boundary value of the first coordinate and nowhere else.
    """

    def __init__(self, core_model: nn.Module, u_in: float, g) -> None:
        super().__init__()
        self.core_model = core_model
        self.u_in = float(u_in)
        self.g = g

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return self.u_in + self.g(X[:, 0:1]) * self.core_model(X)


def train_pinn(model, X_collocation, residual_fn, *, steps=1500,
               learning_rate=3.0e-3, record_every=25):
    """Residual-only training; the boundary is handled by the model itself.

    The cosine schedule decays the learning rate toward the end of training. Two
    input dimensions mean a larger domain to fit, and a large late-stage step
    tends to jitter around the solution rather than settle into it.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=steps, eta_min=learning_rate / 50.0
    )
    history = []
    for step in range(steps):
        optimizer.zero_grad()
        loss = residual_fn(model, X_collocation).square().mean()
        loss.backward()
        optimizer.step()
        scheduler.step()
        if step % record_every == 0 or step == steps - 1:
            history.append(float(loss.detach()))
    return history


def make_grid(a_values: torch.Tensor, b_values: torch.Tensor):
    """Return plotting meshes A, B and the matching (N, 2) input tensor."""
    A, B = torch.meshgrid(a_values, b_values, indexing="ij")
    X = torch.stack([A.reshape(-1), B.reshape(-1)], dim=1)
    return A.numpy(), B.numpy(), X


def plot_field(A, B, values, *, title="", xlabel="", ylabel="", cbar_label="u",
               mask=None, cmap="viridis"):
    """Heat map of a 2D field, with optional masking of an invalid region."""
    Z = np.asarray(values.detach()).reshape(A.shape) if torch.is_tensor(values) \
        else np.asarray(values).reshape(A.shape)
    if mask is not None:
        keep = np.asarray(mask.detach()).reshape(A.shape) if torch.is_tensor(mask) \
            else np.asarray(mask).reshape(A.shape)
        Z = np.where(keep, Z, np.nan)
    fig, ax = plt.subplots(figsize=(6.6, 4.4))
    mesh = ax.pcolormesh(A, B, Z, shading="auto", cmap=cmap)
    fig.colorbar(mesh, ax=ax, label=cbar_label)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    fig.tight_layout()
    return fig, ax


def plot_curves(x, curves, *, title="", xlabel="", ylabel="", styles=None):
    fig, ax = plt.subplots(figsize=(6.6, 4.0))
    for i, (label, y) in enumerate(curves.items()):
        style = "-" if styles is None else styles[i]
        ax.plot(np.asarray(x), np.asarray(y), style, label=label)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(); ax.grid(alpha=0.25)
    fig.tight_layout()
    return fig, ax


print("supplied components ready")

## 2. Path A — a second spatial dimension: spherical geometry

### Why a second coordinate appears

In Notebook 00 the medium was a slab and every ray travelled the same way: straight up, along $+z$. One coordinate was enough.

Now wrap the medium into a **spherical shell** — a star's envelope, or a planetary atmosphere seen as a whole rather than locally. A ray leaving the base of the shell at some angle no longer travels radially. As it moves outward, its direction *relative to the local radial direction* keeps changing, simply because "radially outward" points somewhere different at every point along the ray. The intensity therefore depends on two things: how far out you are, and which way the ray is pointing.

We label the direction with $\theta$, the angle between the ray and the outward radial direction, and work with its cosine

$$
\mu = \cos\theta ,
$$

which is the standard variable in this subject. $\mu = 1$ is a purely radial ray; $\mu = 0$ is a ray travelling tangentially.

### The equation

The transport equation in spherical geometry is

$$
\mu\,\frac{\partial I_\lambda}{\partial r}
\;+\;\frac{1-\mu^{2}}{r}\,\frac{\partial I_\lambda}{\partial \mu}
\;=\;-\rho\kappa_\lambda\left(I_\lambda - S_\lambda\right).
$$

Two things changed relative to Notebook 00, and it is worth separating them:

- the radial derivative picked up a factor $\mu$, because a ray at angle $\theta$ climbs in radius more slowly than it travels;
- an entirely new term $\dfrac{1-\mu^{2}}{r}\dfrac{\partial I_\lambda}{\partial \mu}$ appeared. This is the curvature term. It is not absorption or emission — no physics was added — it is the bookkeeping cost of measuring every ray against a radial direction that rotates underneath it.

Nondimensionalising exactly as you did in Notebook 00, with $r = r_0\hat{r}$ and $I_\lambda = S_0 u$, gives the residual you will implement:

$$
r(\hat{r},\mu)\;=\;\mu\,\frac{\partial u}{\partial \hat{r}}
\;+\;\frac{1-\mu^{2}}{\hat{r}}\,\frac{\partial u}{\partial \mu}
\;+\;\tau\left(u - \texttt{SOURCE}\right)\;=\;0 .
$$

**A useful check before you write any code.** Set $\mu=1$. The factor on the first term becomes 1, the curvature term is multiplied by $1-\mu^2 = 0$ and vanishes, and what is left is

$$
\frac{\partial u}{\partial \hat{r}} + \tau\left(u - \texttt{SOURCE}\right) = 0,
$$

which is precisely the residual from Notebook 00. The 1D problem you already solved is the radial slice of this one. If your 2D implementation disagrees with your 1D one along $\mu=1$, the 2D implementation is wrong.

### The domain, the boundary, and the part the boundary does not reach

The shell runs from its base $\hat{r} = R_{\mathrm{MIN}}$ to its surface $\hat{r} = 1$, and we follow outward-going rays, $\mu \in [0, 1]$. Radiation enters at the base, so the inflow condition is

$$
u(R_{\mathrm{MIN}}, \mu) = u_0 ,
$$

which we impose as a hard constraint with $g(\hat{r}) = \hat{r} - R_{\mathrm{MIN}}$ — the same construction as Notebook 02, adapted to a boundary that now sits at $R_{\mathrm{MIN}}$ instead of at zero.

There is one genuinely new wrinkle, and it is worth taking seriously rather than stepping around, because it is the same well-posedness question Notebook 00 raised in a new disguise. A ray passing through the point $(\hat{r}, \mu)$ has **impact parameter**

$$
p = \hat{r}\sqrt{1-\mu^{2}} ,
$$

the closest distance to the centre it would reach if extended backwards. This is constant along the ray. If $p \le R_{\mathrm{MIN}}$ the ray does come from the base of the shell, and the boundary condition tells us its intensity. If $p > R_{\mathrm{MIN}}$ the ray sails over the base without ever touching it — it entered the shell somewhere else entirely — and our single boundary condition says **nothing** about it.

So the inflow condition determines the solution on the wedge $p \le R_{\mathrm{MIN}}$ and leaves the rest undetermined. We therefore sample collocation points only inside that wedge, and mask the remainder in every plot. This is not a numerical technicality: it is the two-dimensional version of the lesson from Notebook 00 §5. A differential equation plus *a* boundary condition is a complete problem only where that boundary condition actually reaches.

On the wedge there is an exact solution to compare against. Along a ray the equation is just the 1D problem again, with distance measured along the ray rather than along the radius, so

$$
u(\hat{r},\mu) = \texttt{SOURCE} + \left(u_0 - \texttt{SOURCE}\right)e^{-\tau s},
\qquad
s = \hat{r}\mu - \sqrt{R_{\mathrm{MIN}}^{2} - \hat{r}^{2}\left(1-\mu^{2}\right)} ,
$$

where $s$ is the path length travelled since leaving the base. It is supplied below.

In [ ]:
R_MIN = 0.5                 # base of the shell, in units of the outer radius
TAU_SHELL = OPACITY         # reuse your Notebook 00 optical depth


def sample_shell(n_points: int, seed: int) -> torch.Tensor:
    """Collocation points on the wedge reached by rays from the base.

    Sampling the impact parameter p in [0, R_MIN] rather than sampling mu
    directly guarantees every point lies in the well-posed region.
    """
    generator = make_generator(seed)
    r = R_MIN + (1.0 - R_MIN) * torch.rand((n_points, 1), generator=generator)
    p = R_MIN * torch.rand((n_points, 1), generator=generator)
    mu = torch.sqrt(torch.clamp(1.0 - (p / r) ** 2, min=0.0))
    return torch.cat([r, mu], dim=1)


def shell_exact(X: torch.Tensor):
    """Supplied analytic solution, plus a mask marking where it is valid."""
    r, mu = X[:, 0:1], X[:, 1:2]
    inside = R_MIN ** 2 - (r ** 2) * (1.0 - mu ** 2)     # >= 0 on the wedge
    s = r * mu - torch.sqrt(torch.clamp(inside, min=0.0))
    u = SOURCE + (u_0 - SOURCE) * torch.exp(-TAU_SHELL * s)
    return u, inside >= 0.0


X_shell = sample_shell(4000, seed=11)
print(f"collocation points: {tuple(X_shell.shape)}")
print(f"  r_hat range: [{float(X_shell[:, 0].min()):.3f}, {float(X_shell[:, 0].max()):.3f}]")
print(f"  mu    range: [{float(X_shell[:, 1].min()):.3f}, {float(X_shell[:, 1].max()):.3f}]"
      f"  (theta up to {float(torch.rad2deg(torch.arccos(X_shell[:, 1].min()))):.0f} degrees)")

### Exercise A1 — write the two-dimensional residual

Implement the residual above. The only new mechanic is reading two partial derivatives out of one `coordinate_derivative` call: column 0 is $\partial u/\partial\hat{r}$ and column 1 is $\partial u/\partial\mu$, in the same order as the input columns.

In [ ]:
def spherical_residual(model: nn.Module, X: torch.Tensor) -> torch.Tensor:
    """r = mu * du/dr_hat + (1-mu^2)/r_hat * du/dmu + tau * (u - SOURCE)."""
    X_local = X.detach().clone().requires_grad_(True)
    u = model(X_local)

    r_hat = X_local[:, 0:1]
    mu = X_local[:, 1:2]

    # TODO: call coordinate_derivative(u, X_local) and split the result into
    #       du_dr (column 0) and du_dmu (column 1).
    du_dr = None
    du_dmu = None

    # TODO: assemble and return the pointwise residual.
    return None

<details>
<summary><strong>Conceptual hint</strong></summary>

`coordinate_derivative(u, X_local)` returns one tensor with the same shape as `X_local`, so `(N, 2)`. Slicing with `[:, 0:1]` and `[:, 1:2]` keeps the column shape `(N, 1)`, which is what the rest of the expression expects — using `[:, 0]` instead would drop to shape `(N,)` and broadcast into an `(N, N)` result.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

```python
gradients = coordinate_derivative(u, X_local)
du_dr = gradients[:, 0:1]
du_dmu = gradients[:, 1:2]
return mu * du_dr + (1.0 - mu ** 2) / r_hat * du_dmu + TAU_SHELL * (u - SOURCE)
```

</details>

In [ ]:
# Check against the supplied analytic solution before training anything.
X_check = sample_shell(500, seed=7)
u_check, _ = shell_exact(X_check.requires_grad_(True))


class _AnalyticShell(nn.Module):
    def forward(self, X):
        return shell_exact(X)[0]


print("max |residual| on the analytic solution:",
      f"{float(spherical_residual(_AnalyticShell(), X_check).detach().abs().max()):.2e}")
print("(expect something near 1e-6 or smaller; a large value means a wrong term or sign)")

In [ ]:
set_seed(3)
shell_model = HardBoundaryModel(
    MLP(input_dim=2),
    u_in=u_0,
    g=lambda r_hat: r_hat - R_MIN,     # vanishes at the base of the shell
)

shell_history = train_pinn(shell_model, X_shell, spherical_residual, steps=4000)

print(f"final physics loss: {shell_history[-1]:.3e}")
plot_curves(np.arange(len(shell_history)) * 25, {"physics loss": shell_history},
            title="Spherical shell training", xlabel="step", ylabel="mean squared residual")
plt.gca().set_yscale("log")
plt.show()

### Exercise A2 — look at the solution

This is the point of the whole path. A curve could be printed as a table; a field has to be seen.

Build an evaluation grid, evaluate the network and the analytic solution on it, and make three maps: the prediction, the reference, and the absolute error. `make_grid` and `plot_field` are supplied, and `plot_field` takes a `mask` argument — pass the validity mask from `shell_exact` so the undetermined region is left blank.

In [ ]:
r_values = torch.linspace(R_MIN, 1.0, 160)
mu_values = torch.linspace(0.0, 1.0, 160)
R_MESH, MU_MESH, X_grid = make_grid(r_values, mu_values)

u_reference, valid = shell_exact(X_grid)
u_prediction = shell_model(X_grid).detach()

# TODO: plot the PINN prediction over the (r_hat, mu) plane.
#       Use plot_field(R_MESH, MU_MESH, <values>, mask=valid, title=..., xlabel=..., ylabel=...)
#       then plt.show().

# TODO: plot the analytic reference the same way, for comparison.

# TODO: plot the absolute error |prediction - reference|, again masked.
#       A different cmap (for example cmap="magma") makes it read as a different quantity.

# Supplied: the numerical summary that goes with the pictures. It is split by
# distance from the tangent-ray edge of the wedge, for reasons discussed below.
edge_distance = R_MIN ** 2 - (X_grid[:, 0:1] ** 2) * (1.0 - X_grid[:, 1:2] ** 2)
error = (u_prediction - u_reference).abs()
near_edge = valid & (edge_distance < 0.02)
interior = valid & (edge_distance >= 0.02)

print(f"points on the well-posed wedge : {int(valid.sum())} of {valid.numel()}")
print(f"max  |error|, whole wedge      : {float(error[valid].max()):.3e}")
print(f"mean |error|, whole wedge      : {float(error[valid].mean()):.3e}")
print(f"mean |error|, near the edge    : {float(error[near_edge].mean()):.3e}")
print(f"mean |error|, wedge interior   : {float(error[interior].mean()):.3e}")

<details>
<summary><strong>Conceptual hint</strong></summary>

All three maps share the same grid; only the values passed to `plot_field` change. Plot the prediction and the reference on the same colour scale by eye before trusting a difference map — if the two look alike but the error map is large, suspect the mask rather than the model.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

```python
plot_field(R_MESH, MU_MESH, u_prediction, mask=valid,
           title="PINN solution", xlabel=r"$\hat{r}$", ylabel=r"$\mu = \cos\theta$")
plt.show()
```

</details>

### Reading the error map

Your error map should be almost uniformly dark, with one bright ridge running along the slanted edge of the wedge. Both features are worth understanding, because they say different things.

The dark interior is the model working: the mean error away from that edge is a couple of parts in a thousand. The ridge is a different animal, and you cannot train it away. Doubling the number of steps improves the interior mean by roughly a quarter but the peak error by under a tenth, and the ridge is still the dominant feature of the map.

The ridge sits exactly where the tangent-ray boundary is, $p = R_{\mathrm{MIN}}$. Look at the path length in the analytic solution,

$$
s = \hat{r}\mu - \sqrt{R_{\mathrm{MIN}}^{2} - \hat{r}^{2}(1-\mu^{2})},
$$

and differentiate it. The square root sits in a denominator, so as the argument approaches zero at the wedge edge, $\partial s/\partial\mu$ and $\partial s/\partial\hat{r}$ both diverge. The exact solution has **unbounded gradients** along that edge — physically, it is the boundary between rays that graze the base of the shell and rays that miss it entirely, and the intensity turns over sharply across it.

A tanh network is a smooth function. It cannot reproduce an infinite slope, so it rounds the corner off, and the error concentrates in a thin band. In the vocabulary of Notebook 02 this is a **representation** failure — the one category where a bigger or differently structured network is the right response, and the one that neither better sampling nor better optimization will touch.

Notice also that this is a diagnostic point. The mean error over the wedge looks respectable, and it hides the ridge completely. Only the map shows it.

In [ ]:
# Supplied: the mu = 1 slice, checked against your Notebook 00 solution.
r_slice = torch.linspace(R_MIN, 1.0, 200).reshape(-1, 1)
X_radial = torch.cat([r_slice, torch.ones_like(r_slice)], dim=1)

radial_pinn = shell_model(X_radial).detach()
radial_1d = exact_solution_torch(r_slice - R_MIN, TAU_SHELL, SOURCE, u_0)

plot_curves(
    r_slice.flatten(),
    {"2D PINN at mu = 1": radial_pinn.flatten(),
     "Notebook 00 solution": radial_1d.flatten()},
    title="The radial slice is the 1D problem",
    xlabel=r"$\hat{r}$", ylabel="u", styles=["-", "--"],
)
plt.show()

print(f"max difference along mu = 1: {float((radial_pinn - radial_1d).abs().max()):.3e}")

### Interpretation checkpoint — Path A

1. Read your field map along a vertical line at fixed $\hat{r}$. Does $u$ increase or decrease as $\mu$ falls from 1 toward the edge of the wedge, and what geometric fact explains it?
2. The curvature term carries no absorption or emission. Why does dropping it still give the wrong answer for $\mu < 1$?
3. Why does the well-posed wedge narrow as $\hat{r}$ grows, and what would you have to add to the problem to fill in the rest of the plane?
4. The boundary condition is satisfied exactly along the entire base of the shell, for every $\mu$. Using Notebook 02's vocabulary, what does that let you rule out if the field map looks wrong?
5. Increase `steps` to 8000 and rerun the training and the error map. Which error statistic improves substantially and which barely moves? Does that match the classification given above?

## 3. Path B — a wavelength axis: from one measurement to a spectrum

### What is actually being expanded

Everything so far has carried a subscript $\lambda$ and then quietly ignored it. `SOURCE` was a single number because the Planck function was evaluated at one wavelength, and $\tau$ was a single number because the opacity was too. That describes one detector channel.

Telescopes do not return one number. They return a spectrum, and nearly all the physical information is in how the intensity varies *across* wavelength — which is exactly the variation we threw away. So make wavelength an input.

The network becomes $N(\hat{z}, \hat{\lambda})$, where $\hat{\lambda} \in [0,1]$ spans a band around the 300 K peak you found with Wien's law in Notebook 00. Two quantities that used to be constants become functions of it.

**The source function.** $S_\lambda$ is still the Planck function at 300 K, but now evaluated across the band. Normalised by the same $S_0$ you used in Notebook 00 — the Planck function at its peak — the nondimensional source becomes

$$
q(\hat{\lambda}) = \frac{B_\lambda(\hat{\lambda},\,300\,\mathrm{K})}{S_0},
$$

which equals 1 at the peak, exactly recovering `SOURCE = 1`, and falls away on either side. It is no longer a constant, and it is the reason the emergent spectrum has a broad shape at all.

**The opacity.** Real opacity is wildly wavelength-dependent; that structure is what spectroscopy reads. We use the simplest interesting case, one absorption line: a continuum with a narrow Gaussian bump in $\kappa_\lambda$,

$$
\hat{\kappa}(\hat{\lambda}) = 1 + A\,\exp\!\left[-\left(\frac{\hat{\lambda}-\hat{\lambda}_c}{w}\right)^{2}\right],
\qquad
\tau(\hat{\lambda}) = \tau_0\,\hat{\kappa}(\hat{\lambda}).
$$

### The residual

Wavelength is a **parameter, not a direction of propagation**. Photons travel through $\hat{z}$; they do not travel through $\hat{\lambda}$. Nothing couples one wavelength to another, so no $\partial u/\partial\hat{\lambda}$ term appears:

$$
r(\hat{z},\hat{\lambda}) = \frac{\partial u}{\partial \hat{z}}
+ \tau(\hat{\lambda})\left[u - q(\hat{\lambda})\right] = 0 .
$$

This is Notebook 00's residual with two constants promoted to functions of the second input. Contrast that with Path A, where the new coordinate *was* a direction and did produce a new derivative. Adding an input dimension and adding a derivative are separate things, and noticing which you are doing is most of the modelling work.

In effect you are training one network to solve a whole family of 1D problems at once, indexed by $\hat{\lambda}$ — which is also why the analytic solution survives unchanged:

$$
u(\hat{z},\hat{\lambda}) = q(\hat{\lambda}) + \left[u_0 - q(\hat{\lambda})\right]e^{-\tau(\hat{\lambda})\hat{z}} .
$$

We keep the incident radiation grey, $u(0,\hat{\lambda}) = u_0$ for all $\hat{\lambda}$, so the hard constraint is the familiar $g(\hat{z}) = \hat{z}$ from Notebook 02.

In [ ]:
LAMBDA_MIN_CM = 5.0e-4      # 5 microns
LAMBDA_MAX_CM = 2.0e-3      # 20 microns
TEMPERATURE_K = 300.0
WIEN_B = 0.28977719         # cm K

LINE_CENTER = 0.5           # in lambda_hat
LINE_WIDTH = 0.04
LINE_STRENGTH = 9.0
TAU_CONTINUUM = OPACITY     # your Notebook 00 optical depth, away from the line

# Same normalization as Notebook 00: the Planck peak at 300 K.
S_REF = float(blackbody_lambda_cgs(torch.tensor(WIEN_B / TEMPERATURE_K), TEMPERATURE_K))


def lambda_cm(lam_hat: torch.Tensor) -> torch.Tensor:
    """Physical wavelength in cm for a normalized lambda_hat in [0, 1]."""
    return LAMBDA_MIN_CM + lam_hat * (LAMBDA_MAX_CM - LAMBDA_MIN_CM)


def source_ratio(lam_hat: torch.Tensor) -> torch.Tensor:
    """q(lambda_hat): the nondimensional Planck source function."""
    return blackbody_lambda_cgs(lambda_cm(lam_hat), TEMPERATURE_K) / S_REF


def tau_lambda(lam_hat: torch.Tensor) -> torch.Tensor:
    """Optical depth across the band: continuum plus one Gaussian line."""
    kappa_hat = 1.0 + LINE_STRENGTH * torch.exp(
        -((lam_hat - LINE_CENTER) / LINE_WIDTH) ** 2
    )
    return TAU_CONTINUUM * kappa_hat


def spectral_exact(X: torch.Tensor) -> torch.Tensor:
    """Supplied analytic solution of the wavelength-dependent problem."""
    z_hat, lam_hat = X[:, 0:1], X[:, 1:2]
    q = source_ratio(lam_hat)
    return q + (u_0 - q) * torch.exp(-tau_lambda(lam_hat) * z_hat)


lam_axis = torch.linspace(0.0, 1.0, 400).reshape(-1, 1)
plot_curves(lam_axis.flatten(),
            {"source q(lambda_hat)": source_ratio(lam_axis).flatten(),
             "u_0 (incident)": torch.full_like(lam_axis, u_0).flatten()},
            title="Source function across the band", xlabel=r"$\hat{\lambda}$",
            ylabel="nondimensional intensity", styles=["-", ":"])
plt.show()

plot_curves(lam_axis.flatten(), {"tau(lambda_hat)": tau_lambda(lam_axis).flatten()},
            title="Optical depth: continuum plus one line", xlabel=r"$\hat{\lambda}$",
            ylabel=r"$\tau$")
plt.show()

print(f"lambda band            : {LAMBDA_MIN_CM * 1e4:.1f} to {LAMBDA_MAX_CM * 1e4:.1f} microns")
print(f"tau in the continuum   : {float(tau_lambda(torch.tensor([[0.0]]))):.2f}")
print(f"tau in the line core   : {float(tau_lambda(torch.tensor([[LINE_CENTER]]))):.2f}")
print(f"q at the line core     : {float(source_ratio(torch.tensor([[LINE_CENTER]]))):.3f}"
      f"   (compare with the incident u_0 = {u_0})")

### Exercise B1 — write the spectral residual

Only $\hat{z}$ is differentiated. Take column 0 of the gradient and ignore column 1 — the network's dependence on $\hat{\lambda}$ is constrained entirely through the coefficients $\tau(\hat{\lambda})$ and $q(\hat{\lambda})$.

In [ ]:
def spectral_residual(model: nn.Module, X: torch.Tensor) -> torch.Tensor:
    """r = du/dz_hat + tau(lambda_hat) * (u - q(lambda_hat))."""
    X_local = X.detach().clone().requires_grad_(True)
    u = model(X_local)

    lam_hat = X_local[:, 1:2]

    # TODO: take du/dz_hat, which is column 0 of coordinate_derivative(u, X_local).
    du_dz = None

    # TODO: assemble the residual using tau_lambda(lam_hat) and source_ratio(lam_hat).
    return None

<details>
<summary><strong>Conceptual hint</strong></summary>

This is the Notebook 00 residual with `OPACITY` replaced by `tau_lambda(lam_hat)` and `SOURCE` replaced by `source_ratio(lam_hat)`. Both are `(N, 1)` tensors rather than scalars, and they multiply and subtract elementwise exactly as the scalars did.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

```python
du_dz = coordinate_derivative(u, X_local)[:, 0:1]
return du_dz + tau_lambda(lam_hat) * (u - source_ratio(lam_hat))
```

</details>

In [ ]:
# Check against the supplied analytic solution, exactly as in Path A.
class _AnalyticSpectral(nn.Module):
    def forward(self, X):
        return spectral_exact(X)


X_spec_check = torch.rand((500, 2), generator=make_generator(5))
print("max |residual| on the analytic solution:",
      f"{float(spectral_residual(_AnalyticSpectral(), X_spec_check).detach().abs().max()):.2e}")


def sample_band(n_points: int, seed: int) -> torch.Tensor:
    """Uniform collocation over (z_hat, lambda_hat) in [0,1] x [0,1]."""
    return torch.rand((n_points, 2), generator=make_generator(seed))


X_band = sample_band(8000, seed=17)

# A wider network than Path A: the line makes tau vary by a factor of ten across a
# narrow stretch of lambda_hat, and resolving that needs a little more capacity.
set_seed(4)
spectral_model = HardBoundaryModel(MLP(input_dim=2, hidden_dim=48),
                                   u_in=u_0, g=lambda z_hat: z_hat)
spectral_history = train_pinn(spectral_model, X_band, spectral_residual, steps=6000)

print(f"final physics loss: {spectral_history[-1]:.3e}")

### Exercise B2 — produce the spectrum

The field $u(\hat{z},\hat{\lambda})$ is worth mapping, but it is not what an observer has. An observer sits outside the slab and receives only what emerges from the far edge, $\hat{z}=1$. That single line through the field **is the spectrum**, and extracting it is the payoff of this path.

Two short pieces of work: map the field, then take the $\hat{z}=1$ edge and plot it against wavelength.

In [ ]:
z_values = torch.linspace(0.0, 1.0, 160)
lam_values = torch.linspace(0.0, 1.0, 160)
Z_MESH, LAM_MESH, X_field = make_grid(z_values, lam_values)

field_prediction = spectral_model(X_field).detach()
field_reference = spectral_exact(X_field)

# TODO: plot the predicted field over the (z_hat, lambda_hat) plane with plot_field.
#       No mask is needed here: every point is determined by the boundary condition.


def emergent_spectrum(model: nn.Module, lam_hat: torch.Tensor) -> torch.Tensor:
    """Intensity leaving the far edge of the slab, u(z_hat=1, lambda_hat)."""
    # TODO: build an (N, 2) input whose first column is all ones and whose second
    #       column is lam_hat, then evaluate the model on it and detach.
    return None


lam_observed = torch.linspace(0.0, 1.0, 400).reshape(-1, 1)

# TODO: plot the emergent spectrum from the PINN against the analytic spectrum,
#       obtained by calling spectral_exact on the same z_hat = 1 inputs.
#       plot_curves(..., styles=["-", "--"]) keeps the two distinguishable.

<details>
<summary><strong>Conceptual hint</strong></summary>

"Evaluate at $\hat{z}=1$ for many wavelengths" means building an input whose first column is constant and whose second column varies — the opposite pattern to the radial slice you took in Path A, where the second column was held at $\mu=1$.

</details>

<details>
<summary><strong>PyTorch hint</strong></summary>

```python
X_edge = torch.cat([torch.ones_like(lam_hat), lam_hat], dim=1)
return model(X_edge).detach()
```

and then

```python
plot_curves(lam_observed.flatten(),
            {"PINN spectrum": emergent_spectrum(spectral_model, lam_observed).flatten(),
             "analytic": spectral_exact(torch.cat([torch.ones_like(lam_observed),
                                                   lam_observed], dim=1)).flatten()},
            title="Emergent spectrum", xlabel=r"$\hat{\lambda}$",
            ylabel=r"$u(1, \hat{\lambda})$", styles=["-", "--"])
plt.show()
```

</details>

In [ ]:
# Supplied: what the line contributes, isolated by re-running without it.
X_edge = torch.cat([torch.ones_like(lam_observed), lam_observed], dim=1)
with_line = spectral_exact(X_edge)

_saved = LINE_STRENGTH
LINE_STRENGTH = 0.0
continuum_only = spectral_exact(X_edge)
LINE_STRENGTH = _saved

plot_curves(lam_observed.flatten(),
            {"with the line": with_line.flatten(),
             "continuum only": continuum_only.flatten()},
            title="The line is the only difference between these two spectra",
            xlabel=r"$\hat{\lambda}$", ylabel=r"$u(1, \hat{\lambda})$", styles=["-", ":"])
plt.show()

peak = int((with_line - continuum_only).abs().argmax())
print(f"largest departure from the continuum at lambda_hat = {float(lam_observed[peak]):.3f}"
      f"  ({float(lambda_cm(lam_observed[peak])) * 1e4:.2f} microns)")
print(f"  continuum here : {float(continuum_only[peak]):.4f}")
print(f"  with the line  : {float(with_line[peak]):.4f}")

### Interpretation checkpoint — Path B

1. The broad shape of the emergent spectrum is not caused by the line at all. What sets it, and what would change if the medium's temperature were raised to 400 K?
2. Is the line feature in emission or in absorption? Justify it by comparing $u_0$ with $q(\hat{\lambda}_c)$ rather than by looking at the plot.
3. What single change to the setup would flip the line the other way? Relate your answer to a physical observing situation.
4. In the line core $\tau = 20$, well outside the range Notebook 00 derived for the troposphere. Given what Notebook 02 showed about large $\tau$, what would you check before trusting the PINN's line depth — and what would you do about it?
5. Why did this expansion add no new derivative to the residual, while Path A did?

## 4. What the two expansions have in common

Both paths made the same three edits, and it is worth naming them, because they are the edits you will make for any expansion of this kind:

1. **The network gained an input.** `MLP(input_dim=2)`, and nothing else about the architecture changed.
2. **The residual gained something.** In Path A a new coordinate that photons travel along, which produced a new derivative term. In Path B a new coordinate that merely *labels* independent problems, which promoted two constants to functions and produced no derivative at all. Deciding which of these you have is the modelling step; the code follows from it.
3. **The boundary condition had to be re-examined.** Path B's carried over unchanged. Path A's turned out to determine the solution on only part of the domain — the same well-posedness question from Notebook 00 §5, wearing a geometric disguise.

And the output changed character. In one dimension you check a solution by reading numbers. In two you have to look, which is why every exercise here ended in a figure: a field map, a slice through it, an emergent spectrum. The map is how you find a localised error of the kind Notebook 02 taught you to distrust global norms about.

The natural next expansion is to combine them — $u(\hat{r}, \mu, \hat{\lambda})$, a spectrum for every direction through a spherical envelope, which is what a stellar atmosphere code computes. Nothing in the method changes. `input_dim=3`, one more term or one more coefficient, and a harder visualisation problem.